In [1]:
# Camada GOLD
# Definição do modelo dimensional, a partir da tabela Silver:

# 1. Atributos descritivos → vão para DIMENSÕES
# 2. Medidas numéricas → vão para FATO

import sqlite3
import pandas as pd


In [2]:
conexao  = sqlite3.connect(database='db_project_eng_dados')

In [ ]:
# Tabela de Classificação de Dados : Data Dictionary / Metadata 

conexao.execute("""
CREATE TABLE IF NOT EXISTS dict_class (
    nome_tabela TEXT,
    campo TEXT,
    papel TEXT,
    camada TEXT,
    observacao TEXT) """)

conexao.execute("""INSERT INTO dict_class(
    nome_tabela,
    campo,
    papel,
    camada,
    observacao) VALUES 
    ('silver_produtos','id_produto','Chave Natural','Silver','Identificador do produto na origem'),
    ('silver_produtos','nome_produto','Dimensão','Gold', 'Atributo descritivo do produto'),
    ('silver_produtos','tipo_dado','Dimensão','Gold', 'Tipo do dado geoespacial'),
    ('silver_produtos','sistema_coordenadas','Dimensão','Gold', 'Sistema de referência espacial'),
    ('silver_produtos','fornecedor','Dimensão','Gold', 'Fornecedor do dado'),
    ('silver_produtos','resolucao_espacial','Medida','Gold', 'Resolução espacial do produto'),
    ('silver_produtos','area_cobertura_km2','Medida','Gold', 'Área de cobertura em km²'),
    ('silver_produtos','preco','Medida','Gold', 'Valor comercial do produto'),        
    ('silver_produtos','data_bronze','Linhagem','Controle', 'Data de carga na camada Bronze'),  
    ('silver_produtos','data_silver','Linhagem','Controle', 'Data de carga na camada Silver');                                                                                                                                          
                
""")

conexao.commit()


In [ ]:
# Mostragem da tabela de classificação de dados em SQLite:

cursor = conexao.cursor()

cursor.execute("SELECT * FROM dict_class;")
rows = cursor.fetchall()

for row in rows:
    print(row)


('silver_produtos', 'id_produto', 'Chave Natural', 'Silver', 'Identificador do produto na origem')
('silver_produtos', 'nome_produto', 'Dimensão', 'Gold', 'Atributo descritivo do produto')
('silver_produtos', 'tipo_dado', 'Dimensão', 'Gold', 'Tipo do dado geoespacial')
('silver_produtos', 'sistema_coordenadas', 'Dimensão', 'Gold', 'Sistema de referência espacial')
('silver_produtos', 'fornecedor', 'Dimensão', 'Gold', 'Fornecedor do dado')
('silver_produtos', 'resolucao_espacial', 'Medida', 'Gold', 'Resolução espacial do produto')
('silver_produtos', 'area_cobertura_km2', 'Medida', 'Gold', 'Área de cobertura em km²')
('silver_produtos', 'preco', 'Medida', 'Gold', 'Valor comercial do produto')
('silver_produtos', 'data_bronze', 'Linhagem', 'Controle', 'Data de carga na camada Bronze')
('silver_produtos', 'data_silver', 'Linhagem', 'Controle', 'Data de carga na camada Silver')


In [ ]:
## Via Pandas + SQL
df_dict = pd.read_sql("""
SELECT *
FROM dict_class

""", conexao)

df_dict


,nome_tabela,campo,papel,camada,observacao
0,silver_produtos,id_produto,Chave Natural,Silver,Identificador do produto na origem
1,silver_produtos,nome_produto,Dimensão,Gold,Atributo descritivo do produto
2,silver_produtos,tipo_dado,Dimensão,Gold,Tipo do dado geoespacial
3,silver_produtos,sistema_coordenadas,Dimensão,Gold,Sistema de referência espacial
4,silver_produtos,fornecedor,Dimensão,Gold,Fornecedor do dado
5,silver_produtos,resolucao_espacial,Medida,Gold,Resolução espacial do produto
6,silver_produtos,area_cobertura_km2,Medida,Gold,Área de cobertura em km²
7,silver_produtos,preco,Medida,Gold,Valor comercial do produto
8,silver_produtos,data_bronze,Linhagem,Controle,Data de carga na camada Bronze
9,silver_produtos,data_silver,Linhagem,Controle,Data de carga na camada Silver


In [9]:
# Camada Gold
# Tabelas Dimensão
# Dimensão Produto:

conexao.executescript("""
CREATE TABLE IF NOT EXISTS dim_produto (
    sk_produto INTEGER PRIMARY KEY AUTOINCREMENT,
    id_produto INTEGER,
    nome_produto TEXT,
    categoria TEXT,       
    tipo_dado TEXT,
    sistema_coordenadas TEXT,
    formato TEXT,
    fornecedor TEXT );
                
INSERT INTO dim_produto (
    id_produto,
    nome_produto,
    categoria,
    tipo_dado,
    sistema_coordenadas,
    formato,
    fornecedor )
                
SELECT DISTINCT 
    id_produto,
    nome_produto,
    categoria,
    tipo_dado,
    sistema_coordenadas,
    formato,
    fornecedor
FROM silver_produtos;""")                
                


conexao.commit()



In [10]:
pd.read_sql("""
SELECT *
FROM dim_produto;
""", conexao)


,sk_produto,id_produto,nome_produto,categoria,tipo_dado,sistema_coordenadas,formato,fornecedor
0,1,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,SIRGAS 2000 / UTM 23S,Shapefile,GeoMapas Ltda
1,2,2,Modelo Digital de Elevação,MDE,Raster,WGS84,GeoTIFF,INPE
2,3,3,Ortoimagem Urbana São Paulo,Imagem Orbital,Raster,SIRGAS 2000,GeoTIFF,Maxar
3,4,4,Mapa de Drenagem Hidrográfica,Hidrografia,Vetorial,SIRGAS 2000,GeoPackage,ANA
4,5,5,Classificação de Vegetação Cerrado,Vegetação,Raster,WGS84,GeoTIFF,IBGE
5,6,6,Limites Administrativos Municipais,Base Cartográfica,Vetorial,SIRGAS 2000,Shapefile,IBGE
6,7,7,Mapa de Risco de Deslizamento,Análise Ambiental,Vetorial,SIRGAS 2000 / UTM 22S,GeoPackage,Defesa Civil
7,8,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,SIRGAS 2000 / UTM 23S,Shapefile,GeoMapas Ltda
8,9,2,Modelo Digital de Elevação,MDE,Raster,WGS84,GeoTIFF,INPE
9,10,3,Ortoimagem Urbana São Paulo,Imagem Orbital,Raster,SIRGAS 2000,GeoTIFF,Maxar


In [13]:
# Dimensão Tempo:

conexao.executescript("""
CREATE TABLE IF NOT EXISTS dim_tempo (
    sk_tempo INTEGER PRIMARY KEY AUTOINCREMENT,
    data DATE,
    ano INTEGER,
    mes INTEGER,
    dia INTEGER);
                
INSERT INTO dim_tempo (data, ano, mes, dia)
SELECT DISTINCT
    data_aquisicao,
    strftime('%Y', data_aquisicao),
    strftime('%m', data_aquisicao),
    strftime('%d', data_aquisicao)
FROM silver_produtos
WHERE data_aquisicao IS NOT NULL;                                                                                                             
""")

conexao.commit()

In [14]:
pd.read_sql("""
SELECT *
FROM dim_tempo;
""", conexao)

,sk_tempo,data,ano,mes,dia
0,1,2023-06-15,2023,6,15
1,2,2022-11-20,2022,11,20
2,3,2023-02-10,2023,2,10
3,4,2021-08-05,2021,8,5
4,5,2022-09-30,2022,9,30
5,6,2023-01-01,2023,1,1
6,7,2023-07-12,2023,7,12


In [15]:
# Tabela Fato
# Fato Produto Geoespacial

conexao.executescript("""
CREATE TABLE IF NOT EXISTS fato_produto (
sk_produto INTEGER,
sk_tempo INTEGER,
resolucao_espacial REAL,
area_cobertura_km2 REAL,
preco REAL,
data_carga DATE,
FOREIGN KEY (sk_produto) REFERENCES dim_produto(sk_produto),
FOREIGN KEY (sk_tempo) REFERENCES dim_tempo(sk_tempo));

INSERT INTO fato_produto (
    sk_produto,
    sk_tempo,
    resolucao_espacial,
    area_cobertura_km2,
    preco, 
    data_carga)                                                               
SELECT 
    dp.sk_produto,
    dt.sk_tempo,
    sp.resolucao_espacial,
    sp.area_cobertura_km2,
    sp.preco,
    date('now')                                                                                                                
FROM silver_produtos sp
JOIN dim_produto dp
    ON sp.id_produto = dp.id_produto
JOIN dim_tempo dt
    ON sp.data_aquisicao = dt.data;                                                                                
                                                                                                
""")



In [16]:
pd.read_sql("""
SELECT *
FROM fato_produto;
""", conexao)

,sk_produto,sk_tempo,resolucao_espacial,area_cobertura_km2,preco,data_carga
0,1,1,10.0,1500.0,3500.0,2025-12-24
1,8,1,10.0,1500.0,3500.0,2025-12-24
2,15,1,10.0,1500.0,3500.0,2025-12-24
3,2,2,30.0,5000.0,0.0,2025-12-24
4,9,2,30.0,5000.0,0.0,2025-12-24
5,16,2,30.0,5000.0,0.0,2025-12-24
6,3,3,0.5,800.0,12000.0,2025-12-24
7,10,3,0.5,800.0,12000.0,2025-12-24
8,17,3,0.5,800.0,12000.0,2025-12-24
9,4,4,1.0,3200.0,0.0,2025-12-24
